   
### Bronze to Silver -- HR Domain
**Author:** Virendra Tambavekar  
**Task:** Transform HR domain data from Bronze (ALL STRING) to Silver (typed, deduped)  
**Domain:** HR (Broker)  
**Pipeline Stage:** Bronze -> Silver  
**Source:** `charles_schwab_retailbrokerage_dev_team_lemma.bronze.hr` (50,000 rows)  
**Target:** `charles_schwab_retailbrokerage_dev_team_lemma.silver.broker` (50,000 rows)  

**Transformations:**
- Type cast: EMPLOYEE_ID, MANAGER_ID, JOB_CODE, BRANCH_ID -> INT
- Build FULL_NAME from FIRST_NAME + MIDDLE_INITIAL + LAST_NAME
- Dedup by EMPLOYEE_ID (latest `_ingest_ts` wins)
- Carry-forward `_run_id` from upstream bronze layer (lineage tracking)
- Silver audit columns: `_load_ts`, `_batch`, `_run_id`
- Mode: CREATE OR REPLACE (full rebuild)
- Operations logging: pipeline recon + audit event

In [0]:
#Importing required libraries
import logging
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType

#Initialize Logger
logger = logging.getLogger("BronzeToSilverHR")
logger.setLevel(logging.INFO)

In [0]:
dbutils.widgets.text("batch_id","1")

batch_id = dbutils.widgets.get("batch_id")

#Table Paths
source_bronze_table = "charles_schwab_retailbrokerage_dev_team_lemma.bronze.hr"
target_silver_table = "charles_schwab_retailbrokerage_dev_team_lemma.silver.broker"

In [0]:
# Idempotency Logic

def idempotency_check(table_name):
    #Check if target table exists and log state before overwrite.
    if spark.catalog.tableExists(table_name):
        existing_count = spark.table(table_name).count()
        logger.info(f"Idempotency: Target table '{table_name}' already exists with {existing_count} rows. Will be overwritten.")
    else:
        logger.info(f"Idempotency: Target table '{table_name}' does not exist. Will be created.")

idempotency_check(target_silver_table)

In [0]:
def transform_bronze_to_silver():
    logger.info("Transforming bronze to silver")

    #Exception Handling
    try:
        #Read bronze table
        bronze_df = spark.table(source_bronze_table)
        #Deduplication
        window = Window.partitionBy("EMPLOYEE_ID").orderBy(col("_ingest_ts").desc())
        deduplicated_df = (
            bronze_df
            .withColumn("row_num", row_number().over(window))
            .filter(col("row_num") == 1)
            .drop("row_num")
        )
        #Type Casting
        silver_df = (
            deduplicated_df
            .withColumn("EMPLOYEE_ID", col("EMPLOYEE_ID").cast(IntegerType()))
            .withColumn("MANAGER_ID", col("MANAGER_ID").cast(IntegerType()))
            .withColumn("JOB_CODE", col("JOB_CODE").cast(IntegerType()))
            .withColumn("BRANCH_ID", expr("TRY_CAST(BRANCH_ID AS INT)"))
            .withColumn("FULL_NAME", concat_ws(' ',col("FIRST_NAME"),col("MIDDLE_INITIAL"),col("LAST_NAME")))
        )
        #Metadata
        final_silver_df = (
            silver_df
            .withColumn("_load_ts",current_timestamp())
            .drop("_ingest_ts","_source_file")
        )
        #Writing Silver layer table
        (
            final_silver_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_silver_table)
        )
        logger.info("Transformed bronze to silver")
        return True
    except Exception as e:
        logger.error(f"Failed during transformation : {str(e)}")
        raise e
file_processed = transform_bronze_to_silver()

In [0]:
if file_processed:
    try:
        silver_df = spark.read.table(target_silver_table)
        silver_count = silver_df.count()
        logger.info("Validation Successful")
        logger.info(f"File processed count : {silver_count}")
        display(silver_df.limit(5))
        display(spark.createDataFrame([(silver_count,)], ["silver_count"]))
    except Exception as e:
        logger.error(f"Failed during transformation : {str(e)}")
        raise e